# ❤️ Proyecto: Predicción de Enfermedades Cardiovasculares

## 🎯 Objetivos del Proyecto

En este proyecto práctico aprenderás a:

1. **Trabajar con datos médicos reales** del UCI Heart Disease Dataset
2. **Construir un pipeline completo** de Machine Learning
3. **Evaluar modelos de clasificación binaria** con múltiples métricas
4. **Comparar diferentes algoritmos** (SGD vs Random Forest)
5. **Registrar experimentos** con MLflow de forma profesional
6. **Interpretar resultados médicos** con responsabilidad

---

## ❤️ ¿Por qué es importante?

Las **Enfermedades Cardiovasculares (ECV)** son la principal causa de muerte en el mundo:

- 💔 Causan **17.9 millones** de muertes al año
- ⚠️ **90% son prevenibles** con detección temprana
- 🏥 Un diagnóstico temprano puede **salvar vidas**
- 🤖 La IA puede ayudar a **detectar patrones** que salven vidas

---

## 📊 El Dataset: UCI Heart Disease

- **303 pacientes** con datos clínicos
- **14 características** médicas (edad, presión arterial, colesterol, etc.)
- **Variable objetivo**: Presencia de enfermedad cardíaca (0/1)
- **Tipo de problema**: Clasificación binaria

---

## 🎓 Ejercicio Especial: Completa los Comentarios

**🚨 IMPORTANTE**: A lo largo de este notebook verás comentarios como:

```python
# TODO: [Completa aquí] ¿Qué hace esta función?
```

**Tu misión** es completar estos comentarios explicando:
- ¿Qué hace el código?
- ¿Por qué es importante?
- ¿Qué resultado esperamos?

Esto te ayudará a:
- ✅ Entender profundamente cada paso
- ✅ Practicar documentación de código
- ✅ Prepararte para proyectos reales

---

## 🚀 ¡Empecemos!

## ⚙️ Paso 2: Configuración de MLflow

Configuramos el experimento donde registraremos todos nuestros entrenamientos.

## 📚 Paso 3: Importar Librerías

Importamos todas las herramientas necesarias para el proyecto.

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# ~> Debemos sustituir el email vacío por nuestro email real de Databricks
email = 'dalonsogomez.ai@gmail.com'  # Email de Databricks

# Validación
if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
else:
    # ~> set_tracking_uri apunta MLflow al servidor de Databricks para registrar experimentos
    mlflow.set_tracking_uri("databricks")
    
    # ~> set_experiment crea (si no existe) y selecciona el experimento donde se agruparán los runs
    experiment_name = f"/Users/{email}/5-prediccion-infarto"
    mlflow.set_experiment(experiment_name)
    
    print("=" * 70)
    print("✅ MLflow configurado correctamente")
    print("=" * 70)
    print(f"📊 Experimento: {experiment_name}")
    print(f"❤️  Proyecto: Predicción de Enfermedades Cardiovasculares")
    print("=" * 70)

In [0]:
# Librerías básicas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ~> train_test_split divide el dataset en conjuntos de entrenamiento y prueba
from sklearn.model_selection import train_test_split

# ~> Un Pipeline encadena pasos de preprocesamiento y el modelo final en un solo objeto
from sklearn.pipeline import Pipeline

# Preprocesamiento
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# ~> Usaremos SGDClassifier (baseline lineal) y RandomForestClassifier (ensemble más potente)
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier

# ~> cross_val_score evalúa el modelo con validación cruzada (K particiones)
from sklearn.model_selection import cross_val_score, cross_val_predict

# ~> Usaremos precision, recall, f1, roc_auc, accuracy y matrices de confusión
# ~> En medicina priorizamos recall para no dejar pasar falsos negativos
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    ConfusionMatrixDisplay, 
    roc_auc_score,
    accuracy_score,
    classification_report,
    roc_curve
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")

## ❤️ Contexto del Proyecto

### El Problema

En este proyecto trabajaremos con el **UCI Heart Disease Dataset**, uno de los datasets más importantes en medicina predictiva. 

### 📊 Datos Clave sobre ECV

- 💔 **Principal causa de muerte** a nivel mundial
- 📈 **17.9 millones** de muertes anuales
- ⚠️ **90% son prevenibles** con detección temprana
- 🎯 **Diagnóstico temprano** = vidas salvadas
- 🤖 **IA** puede detectar patrones imperceptibles para humanos

### 🎯 Nuestro Objetivo

Construir un modelo de Machine Learning que pueda **predecir la presencia de enfermedad cardiovascular** basándose en características clínicas del paciente.

### ⚕️ Responsabilidad Ética

> ⚠️ **IMPORTANTE**: Este es un proyecto educativo. Los modelos médicos reales requieren:
> - Validación clínica exhaustiva
> - Aprobación regulatoria
> - Supervisión médica profesional
> - Consideraciones éticas y legales

Nuestro objetivo es aprender las técnicas, no reemplazar el criterio médico.

## 📚 Fundamentos: Clasificación Binaria

### 🎯 ¿Qué es Clasificación Binaria?

Un problema donde el modelo debe elegir entre **dos clases**:
- ✅ Clase Positiva (1): Tiene enfermedad cardíaca
- ❌ Clase Negativa (0): No tiene enfermedad cardíaca

---

### 🔄 Pipeline de Entrenamiento

```
1. Preparación de Datos
   ↓
2. Selección del Modelo
   ↓
3. Entrenamiento
   ↓
4. Ajuste de Hiperparámetros
   ↓
5. Evaluación
```

---

### 📊 Métricas de Evaluación

#### 1. **Matriz de Confusión**

|                    | Predicción: No (0) | Predicción: Sí (1) |
|--------------------|--------------------|--------------------|
| **Real: No (0)**   | TN (Verdadero Neg) | FP (Falso Positivo)|
| **Real: Sí (1)**   | FN (Falso Negativo)| TP (Verdadero Pos) |

#### 2. **Precision (Precisión)**
```
Precision = TP / (TP + FP)
```
- "De los que predije como enfermos, ¿cuántos realmente lo están?"
- **Importante cuando**: Los falsos positivos son costosos

#### 3. **Recall (Sensibilidad)**
```
Recall = TP / (TP + FN)
```
- "De todos los enfermos reales, ¿cuántos detecté?"
- **Importante cuando**: No podemos perder ningún caso positivo (medicina)

#### 4. **F1-Score**
```
F1 = 2 × (Precision × Recall) / (Precision + Recall)
```
- Balance entre Precision y Recall
- Útil cuando las clases están desbalanceadas

#### 5. **ROC-AUC**
- Curva ROC: Representa el trade-off entre True Positive Rate y False Positive Rate
- AUC: Área bajo la curva (0.5 = aleatorio, 1.0 = perfecto)

#### 6. **Validación Cruzada**
- Divide datos en K partes (folds)
- Entrena K veces, cada vez con un fold diferente como test
- Promedia resultados para obtener estimación robusta

---

### 🎯 ¿Qué métrica usar?

| Situación | Métrica Principal |
|-----------|-------------------|
| Clases balanceadas | Accuracy |
| No perder positivos (medicina) | **Recall** ⭐ |
| Evitar falsos positivos | Precision |
| Balance general | F1-Score |
| Comparar modelos | ROC-AUC |

**En medicina, típicamente priorizamos Recall** porque es mejor detectar un caso falso positivo que perder un verdadero positivo.

## 📁 Paso 4: Cargar y Explorar los Datos

Cargamos el dataset y realizamos un análisis exploratorio inicial.

In [0]:
# -> Cargar el dataset Heart Disease desde el archivo CSV

# ~> Leemos los datos desde la ruta relativa del proyecto
data = pd.read_csv('/Workspace/Users/dalonsogomez.ai@gmail.com/clase_mlops_mlflow-202637904/data/raw/heart.csv')

# > Mostrar número de filas, columnas y variable objetivo
print("=" * 60)
print("INFORMACIÓN DEL DATASET HEART DISEASE")
print("=" * 60)
print(f"Número de muestras: {data.shape[0]}")
print(f"Número de características: {data.shape[1] - 1}  (excluyendo target)")
print(f"Variable objetivo: target (0 = sin enfermedad, 1 = con enfermedad)")
print("=" * 60)

# => Dataset cargado: 303 pacientes, 13 características + 1 target binario


In [0]:
# ~> Mostrar las primeras filas para inspeccionar tipos de datos y valores
# ~> Esta inspección es útil para detectar columnas categóricas, nulos y rangos antes de modelar
display(data.head())


### 📋 Diccionario de Datos

Cada columna representa una característica médica importante:

#### 👤 Datos Demográficos
1. **age**: Edad del paciente en años
2. **sex**: Sexo (1 = masculino, 0 = femenino)

#### 💊 Síntomas y Diagnóstico
3. **cp**: Tipo de dolor de pecho (Chest Pain)
   - 0: Angina típica
   - 1: Angina atípica
   - 2: Dolor no anginoso
   - 3: Asintomático

4. **exang**: Angina inducida por ejercicio (1 = sí, 0 = no)

#### 🩺 Mediciones Clínicas
5. **trestbps**: Presión arterial en reposo (mm Hg)
6. **chol**: Colesterol sérico (mg/dl)
7. **fbs**: Azúcar en sangre en ayunas > 120 mg/dl (1 = verdadero, 0 = falso)
8. **thalach**: Frecuencia cardíaca máxima alcanzada

#### 📊 Resultados de Pruebas
9. **restecg**: Resultados electrocardiográficos en reposo
   - 0: Normal
   - 1: Anormalidad de onda ST-T
   - 2: Hipertrofia ventricular izquierda

10. **oldpeak**: Depresión del segmento ST inducida por ejercicio

11. **slope**: Pendiente del segmento ST durante ejercicio
    - 0: Ascendente
    - 1: Plano
    - 2: Descendente

12. **ca**: Número de vasos principales coloreados por fluoroscopia (0-3)

13. **thal**: Resultados de prueba de talasemia
    - 1: Normal
    - 2: Defecto fijo
    - 3: Defecto reversible

#### 🎯 Variable Objetivo
14. **target**: Presencia de enfermedad cardíaca
    - 0: No enfermedad
    - 1: Enfermedad presente

---

**💡 Tip**: En medicina, cada una de estas características tiene significado clínico. Un buen data scientist debe entender el dominio del problema.

In [0]:
# ~> Revisar tipos de datos y valores nulos con info()
data.info()

# >> Observaciones:
# ~> Columnas categóricas (object/bool): sex, cp, fbs, restecg, exang, slope, thal
# ~> Columnas numéricas (float64/int64): age, trestbps, chol, thalach, oldpeak, ca, target
# ~> Hay valores nulos en trestbps y chol (Non-Null Count < 303)


In [0]:
# ~> Obtener estadísticas descriptivas de las columnas numéricas
display(data.describe())

# >> Observaciones sobre rangos, medias y valores atípicos:
# ~> age: media ~54 años, rango [29, 77]
# ~> trestbps: presión arterial, rango [80, 200], con nulos
# ~> chol: colesterol, rango [126, 564], con nulos
# ~> thalach: frecuencia cardíaca máxima, rango [71, 202]
# ~> oldpeak: depresión ST, rango [0, 6.2], la mayoría cerca de 0
# ~> ca: vasos coloreados (0-3), discreta
# ~> Las escalas son muy diferentes entre columnas, necesitarán normalización


### 💡 Observaciones Clave

**~> Basándote en las estadísticas:**
- La edad media de los pacientes es ~54 años, con un rango de 29 a 77
- El rango de valores varía mucho entre características (age: 29-77 vs oldpeak: 0-6.2)
- Hay valores nulos en trestbps y chol que el pipeline imputará con la mediana
- Las características numéricas necesitarán normalización (StandardScaler) por sus diferentes escalas

In [0]:
# -> Analizar la distribución de la variable objetivo

# ~> Contar muestras por clase
target_counts = data.target.value_counts()
print("=" * 60)
print("DISTRIBUCIÓN DE LA VARIABLE OBJETIVO")
print("=" * 60)
print(target_counts)
print(f"\nProporción clase 0 (sin enfermedad): {target_counts[0]/len(data)*100:.1f}%")
print(f"Proporción clase 1 (con enfermedad):  {target_counts[1]/len(data)*100:.1f}%")
print("=" * 60)

# ~> El dataset está relativamente balanceado (165 vs 138)
# ~> Un dataset balanceado es importante para que el modelo no sesgue hacia una clase
# ~> Si estuviera desbalanceado, habríamos necesario usar class_weight o técnicas de resampling

# => Dataset balanceado: 54.5% clase 1, 45.5% clase 0


### 🔍 Análisis Exploratorio Visual

Visualicemos las relaciones entre características para entender mejor los datos.

In [0]:
# -> Crear visualizaciones exploratorias del dataset Heart Disease

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Análisis Exploratorio - Heart Disease Dataset', fontsize=16, fontweight='bold')

# > Gráfico 1: Distribución de edad por diagnóstico
sns.histplot(data=data, x='age', hue='target', kde=True, ax=axes[0, 0], 
             palette='Set2', alpha=0.6)
axes[0, 0].set_xlabel('Edad (años)', fontsize=11)
axes[0, 0].set_ylabel('Frecuencia', fontsize=11)
axes[0, 0].set_title('Distribución de Edad por Diagnóstico', fontweight='bold')

# > Gráfico 2: Colesterol vs Presión arterial coloreado por target
scatter = axes[0, 1].scatter(data['chol'], data['trestbps'], c=data['target'], 
                              cmap='Set2', alpha=0.7, edgecolors='k')
axes[0, 1].set_xlabel('Colesterol (mg/dl)', fontsize=11)
axes[0, 1].set_ylabel('Presión Arterial (mm Hg)', fontsize=11)
axes[0, 1].set_title('Colesterol vs Presión Arterial', fontweight='bold')
legend1 = axes[0, 1].legend(*scatter.legend_elements(), title='Target')
axes[0, 1].add_artist(legend1)

# > Gráfico 3: Tipo de dolor de pecho vs diagnóstico
cp_target = data.groupby(['cp', 'target']).size().unstack()
cp_target.plot(kind='bar', ax=axes[1, 0], color=['#66c2a5', '#fc8d62'])
axes[1, 0].set_xlabel('Tipo de Dolor de Pecho', fontsize=11)
axes[1, 0].set_ylabel('Número de Pacientes', fontsize=11)
axes[1, 0].set_title('Dolor de Pecho vs Diagnóstico', fontweight='bold')
axes[1, 0].legend(title='Target', labels=['Sin enfermedad', 'Con enfermedad'])
axes[1, 0].tick_params(axis='x', rotation=45)

# > Gráfico 4: Matriz de correlación
corr = data.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[1, 1],
            fmt='.2f', square=True, linewidths=0.5, annot_kws={'size': 7})
axes[1, 1].set_title('Matriz de Correlación', fontweight='bold')

plt.tight_layout()
plt.show()

# >> Patrones observados en los gráficos:
# ~> Los pacientes con enfermedad cardíaca tienden a ser algo más jóvenes
# ~> El dolor asintomático (asymptomatic) se asocia con mayor probabilidad de enfermedad
# ~> cp, thalach, oldpeak y ca muestran las correlaciones más fuertes con target
# ~> No hay una separación clara entre colesterol/presión arterial y target

# => Visualizaciones generadas: el dolor asintomático y la edad son los predictores más visuales


## 🔀 Paso 5: Dividir los Datos

Dividimos el dataset en conjuntos de entrenamiento y prueba.

In [0]:
# -> Dividir los datos en conjuntos de entrenamiento y prueba

# ~> Usamos train_test_split con test_size=0.2 y random_state=5 (igual que la solución)
train_set, test_set = train_test_split(data, test_size=0.2, random_state=5)

# > Comprobar el tamaño de cada conjunto
print("=" * 60)
print("DIVISIÓN DE DATOS")
print("=" * 60)
print(f"Datos totales: {len(data)} muestras")
print(f"\nConjunto de Entrenamiento:")
print(f"   - Muestras: {len(train_set)} ({len(train_set)/len(data)*100:.1f}%)")
print(f"   - Shape: {train_set.shape}")
print(f"\nConjunto de Prueba:")
print(f"   - Muestras: {len(test_set)} ({len(test_set)/len(data)*100:.1f}%)")
print(f"   - Shape: {test_set.shape}")
print("=" * 60)
print("Datos preparados para el preprocesamiento")

# => 242 muestras de train (80%), 61 de test (20%)


In [0]:
# ~> Generar histogramas del conjunto de entrenamiento para ver distribuciones
train_set.hist(bins=50, figsize=(20, 15))
plt.show()

# >> Características con escalas o distribuciones muy diferentes:
# ~> age: rango [29, 77], distribución aprox. normal
# ~> trestbps: rango [80, 200], sesgada a la derecha
# ~> chol: rango [126, 564], algunos valores atípicos altos
# ~> thalach: rango [71, 202], distribución aprox. normal invertida
# ~> oldpeak: mayoría concentrada en 0-1, sesgada
# ~> ca: discreta (0, 1, 2, 3)

# => Las escalas son muy diferentes, StandardScaler será necesario para SGD


### 📏 Observación Importante: Escalas Diferentes

**~> Las características tienen escalas muy diferentes (age: 29-77, chol: 126-564, oldpeak: 0-6.2), lo que hace que las de mayor rango dominen el cálculo de distancia en modelos sensibles a escala como SGD.**

Revisa columnas como `age`, `trestbps`, `chol`, `thalach` y `oldpeak`.

**~> Aplicaríamos StandardScaler para estandarizar cada variable numérica (media=0, desviación=1), haciéndolas comparables.**

**~> Esto ayuda a modelos basados en gradiente (SGD) o distancia (KNN, SVM) a converger más rápido y tratar todas las características con la misma importancia.**


## 🔧 Paso 6: Construir el Pipeline de Preprocesamiento

Crearemos un pipeline que prepare los datos automáticamente.

In [0]:
# -> Clasificar columnas en categóricas y numéricas

# ~> Columnas categóricas: variables con texto o categorías
# ~> One-Hot Encoding crea una columna binaria por cada categoría, evitando ordinales artificiales
cat_attr = ["sex", "cp", "fbs", "restecg", "exang", "slope"]

# ~> Columnas numéricas: variables continuas o discretas
num_attr = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca", "thal"]

# ~> Pipeline para variables numéricas: imputar nulos con mediana + estandarizar
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("std_scaler", StandardScaler())
])

# ~> ColumnTransformer que combina numéricas y categóricas
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attr),
    ("cat", OneHotEncoder(), cat_attr)
])

# ~> Usamos One-Hot Encoding para categóricas porque no hay orden intrínseco
# ~> entre categorías como 'asymptomatic' y 'typical angina'

# => Pipeline de preprocesamiento listo: imputa nulos, escala numéricas y codifica categóricas


## 🎯 Paso 7: Preparar X e y

Separamos características (X) de la variable objetivo (y).

In [0]:
# ~> Separar características (X) y target (y) del conjunto de entrenamiento
x_train = train_set.drop("target", axis=1)
y_train = train_set.target

# > Comprobar dimensiones resultantes
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Columnas en x_train: {list(x_train.columns)}")


In [0]:
# ~> Aplicar el pipeline al conjunto de entrenamiento
# ~> fit_transform calcula los parámetros (media, std, categorías) y transforma los datos
# ~> transform solo aplica los parámetros ya calculados (se usa en test)
x_train_pr = full_pipeline.fit_transform(x_train)

# > Comprobar cómo cambia el número de columnas tras One-Hot Encoding
print(f"x_train original: {x_train.shape[1]} columnas")
print(f"x_train procesado: {x_train_pr.shape[1]} columnas")
print(f"\n>> El One-Hot Encoding expandió las categóricas en múltiples columnas binarias")


## 🤖 Paso 8: Entrenamiento y Evaluación de Modelos

Entrenaremos dos modelos y compararemos sus resultados usando MLflow.

### 🔵 Modelo 1: SGD Classifier (Baseline)

**~> SGD (Stochastic Gradient Descent) es un algoritmo de optimización que actualiza los parámetros del modelo gradualmente usando pequeños lotes de datos. En sklearn, SGDClassifier usa descenso de gradiente para encontrar el hiperplano óptimo que separa las clases.**

Empezaremos con un modelo simple como baseline (línea base) para tener una referencia.

In [0]:
# ========================================
# -> ENTRENAMIENTO SGD CLASSIFIER - BASELINE
# ========================================

# ~> Activar autolog de MLflow para registrar parámetros y métricas automáticamente
mlflow.sklearn.autolog()

# ~> Iniciar un run para el modelo baseline SGD
with mlflow.start_run(run_name="SGD Classifier - Baseline") as run:
    
    print("=" * 70)
    print("INICIANDO ENTRENAMIENTO SGD CLASSIFIER")
    print("=" * 70)
    
    # > Crear SGDClassifier con semilla fija
    sgd_clf = SGDClassifier(random_state=42)
    
    # > Evaluar con cross_val_score (3 folds)
    scores = cross_val_score(sgd_clf, x_train_pr, y_train, cv=3, scoring="accuracy")
    
    print(f"\nCross-Validation Scores (3-fold): {scores}")
    print(f"Media de accuracy: {scores.mean():.4f}")
    print(f"Desviación estándar: {scores.std():.4f}")
    
    # > Registrar métricas de validación cruzada en MLflow
    mlflow.log_metric("cv_accuracy", scores.mean())
    mlflow.log_metric("cv_std", scores.std())
    
    print("\nMétricas registradas en MLflow")
    print("=" * 70)
    
    # ~> El accuracy de ~0.76 no es malo, pero podemos mejorar evaluando con otras métricas

mlflow.end_run()
print("\nRun finalizado correctamente")

# => SGD baseline: accuracy CV ~0.76, sirve como referencia para comparar con Random Forest


### 📊 Evaluación del Modelo SGD

In [0]:
# ~> Obtener predicciones con validación cruzada
# ~> cross_val_predict es útil porque devuelve predicciones de cada muestra
# ~> cuando esa muestra estaba en el fold de prueba, sin entrenar sobre ella
preds = cross_val_predict(sgd_clf, x_train_pr, y_train, cv=3)

# >> Predicciones generadas para construir métricas y matriz de confusión
print(f"Predicciones generadas: {len(preds)} muestras")


In [0]:
# ~> Calcular y visualizar la matriz de confusión del modelo SGD
cm = confusion_matrix(y_train, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Sin enfermedad', 'Con enfermedad'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusión - SGD Classifier', fontweight='bold')
plt.show()

# >> Interpretación TN, FP, FN, TP:
# ~> TN (Verdaderos Negativos): pacientes sin enfermedad correctamente clasificados
# ~> FP (Falsos Positivos): pacientes sin enfermedad clasificados como enfermos
# ~> FN (Falsos Negativos): pacientes con enfermedad NO detectados ← MÁS GRAVE en medicina
# ~> TP (Verdaderos Positivos): pacientes con enfermedad correctamente detectados

# ~> En medicina, los Falsos Negativos son más graves porque un paciente enfermo
# ~> no recibiría tratamiento al ser clasificado incorrectamente como sano


In [0]:
# ~> Calcular métricas del modelo SGD
precision = precision_score(y_train, preds)
recall = recall_score(y_train, preds)
f1 = f1_score(y_train, preds)
roc_auc = roc_auc_score(y_train, preds)

# > Imprimir e interpretar las métricas
print("=" * 60)
print("MÉTRICAS DEL MODELO SGD CLASSIFIER")
print("=" * 60)
print(f"Precision:  {precision:.4f}  (de los predichos como enfermos, ¿cuántos lo están?)")
print(f"Recall:     {recall:.4f}  (de los enfermos reales, ¿cuántos detectamos?)")
print(f"F1-Score:   {f1:.4f}  (balance entre precision y recall)")
print(f"ROC-AUC:    {roc_auc:.4f}  (capacidad de discriminación)")
print("=" * 60)

# ~> En medicina, Recall es la métrica más importante porque es mejor detectar
# ~> un falso positivo (que se aclara con más pruebas) que perder un enfermo real

# => SGD: Precision ~0.76, Recall ~0.83, F1 ~0.79, ROC-AUC ~0.75


### 🌲 Modelo 2: Random Forest Classifier

**~> Random Forest es un ensemble de múltiples árboles de decisión que votan conjuntamente. Suele funcionar mejor que modelos simples porque reduce la varianza mediante el promedio de múltiples árboles entrenados con diferentes subconjuntos de datos y características.**

Ahora probemos un modelo más potente y comparemos resultados.

In [0]:
# ========================================
# -> ENTRENAMIENTO RANDOM FOREST CLASSIFIER
# ========================================

# ~> Iniciar un nuevo run para Random Forest
with mlflow.start_run(run_name="Random Forest Classifier") as run:
    
    print("=" * 70)
    print("INICIANDO ENTRENAMIENTO RANDOM FOREST")
    print("=" * 70)
    
    # > Crear y evaluar Random Forest con cross-validation
    rf_clf = RandomForestClassifier(random_state=42)
    rf_scores = cross_val_score(rf_clf, x_train_pr, y_train, cv=3, scoring="accuracy")
    rf_preds = cross_val_predict(rf_clf, x_train_pr, y_train, cv=3)
    
    print(f"\nCross-Validation Scores (3-fold): {rf_scores}")
    print(f"Media de accuracy: {rf_scores.mean():.4f}")
    print(f"Desviación estándar: {rf_scores.std():.4f}")
    
    mlflow.log_metric("cv_accuracy", rf_scores.mean())
    mlflow.log_metric("cv_std", rf_scores.std())
    
    print("=" * 70)

# ~> Hiperparámetros que se podrían ajustar: n_estimators, max_depth, max_features
# ~> Comparamos contra SGD para ver si un modelo más complejo mejora el rendimiento

# => Random Forest: accuracy CV mejor que SGD, ahora evaluamos en detalle


In [0]:
# ~> Calcular y visualizar la matriz de confusión de Random Forest
cm_rf = confusion_matrix(y_train, rf_preds)
disp_rf = ConfusionMatrixDisplay(cm_rf, display_labels=['Sin enfermedad', 'Con enfermedad'])
disp_rf.plot(cmap='Greens')
plt.title('Matriz de Confusión - Random Forest', fontweight='bold')
plt.show()

# ~> Comparar los errores con los del modelo SGD:
# ~> Random Forest debería tener menos falsos negativos que SGD
# ~> Esto es crucial en medicina donde los FN son los errores más graves


In [0]:
# ~> Calcular métricas de Random Forest
rf_precision = precision_score(y_train, rf_preds)
rf_recall = recall_score(y_train, rf_preds)
rf_f1 = f1_score(y_train, rf_preds)
rf_roc_auc = roc_auc_score(y_train, rf_preds)

# > Crear tabla comparativa SGD vs Random Forest
print("=" * 60)
print("COMPARACIÓN DE MODELOS")
print("=" * 60)
print(f"{'Métrica':<15} {'SGD':>10} {'Random Forest':>15}")
print("-" * 45)
print(f"{'Precision':<15} {precision:>10.4f} {rf_precision:>15.4f}")
print(f"{'Recall':<15} {recall:>10.4f} {rf_recall:>15.4f}")
print(f"{'F1-Score':<15} {f1:>10.4f} {rf_f1:>15.4f}")
print(f"{'ROC-AUC':<15} {roc_auc:>10.4f} {rf_roc_auc:>15.4f}")
print("=" * 60)

# ~> Random Forest funciona mejor que SGD en todas las métricas
# ~> El ensemble reduce la varianza y captura relaciones no lineales

# => Random Forest supera a SGD en todas las métricas, especialmente en F1 y ROC-AUC


## 🎯 Paso 9: Entrenamiento Final y Evaluación en Test

Ahora entrenaremos el modelo con **todo el conjunto de entrenamiento** y evaluaremos en el conjunto de prueba (que nunca ha visto).

In [0]:
# ~> Entrenar el modelo final usando todo el conjunto de entrenamiento procesado
# ~> Entrenamos con todos los datos de train para aprovechar toda la información disponible
# ~> La evaluación final se hará en el conjunto de prueba que el modelo nunca ha visto
forest_clf = RandomForestClassifier(random_state=42)
forest_clf.fit(x_train_pr, y_train)

print("Modelo final entrenado con todo el conjunto de entrenamiento")
print(f"   - Muestras de entrenamiento: {x_train_pr.shape[0]}")
print(f"   - Características procesadas: {x_train_pr.shape[1]}")


In [0]:
# ~> Separar características y target del conjunto de prueba
x_test = test_set.drop("target", axis=1)
y_test = test_set.target

# > Comprobar dimensiones
print(f"x_test shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")


In [0]:
# ~> Transformar el conjunto de prueba y generar predicciones
# ~> Usamos transform() y NO fit_transform() porque los parámetros del pipeline
# ~> (media, std, categorías) ya fueron calculados con train y no deben contaminarse con test
x_test_pr = full_pipeline.transform(x_test)
final_preds = forest_clf.predict(x_test_pr)

print("Conjunto de prueba transformado y predicciones generadas")


In [0]:
# ~> Evaluar resultados finales en el conjunto de prueba
final_precision = precision_score(y_test, final_preds)
final_recall = recall_score(y_test, final_preds)
final_f1 = f1_score(y_test, final_preds)
final_roc_auc = roc_auc_score(y_test, final_preds)
final_accuracy = accuracy_score(y_test, final_preds)

print("=" * 70)
print("EVALUACIÓN FINAL EN CONJUNTO DE PRUEBA")
print("=" * 70)
print(f"Accuracy:   {final_accuracy:.4f}")
print(f"Precision:  {final_precision:.4f}")
print(f"Recall:     {final_recall:.4f}  (métrica clave en medicina)")
print(f"F1-Score:   {final_f1:.4f}")
print(f"ROC-AUC:    {final_roc_auc:.4f}")
print("=" * 70)

# > Visualizar la matriz de confusión final
cm_final = confusion_matrix(y_test, final_preds)
disp_final = ConfusionMatrixDisplay(cm_final, display_labels=['Sin enfermedad', 'Con enfermedad'])
disp_final.plot(cmap='Purples')
plt.title('Matriz de Confusión Final - Test Set', fontweight='bold')
plt.show()

# > Generar classification report
print("\nClassification Report:")
print(classification_report(y_test, final_preds, target_names=['Sin enfermedad', 'Con enfermedad']))

# ~> El modelo generaliza bien si las métricas en test son similares a las de validación cruzada
# ~> Un recall alto significa que detectamos la mayoría de pacientes enfermos

# => Modelo final: Recall ~0.94, F1 ~0.91, ROC-AUC ~0.90 en el conjunto de prueba


## 🎉 Reflexión Final del Proyecto

Completa esta sección cuando termines el notebook.

### ✅ Comprueba que has trabajado

1. [x] Exploración de datos médicos reales
2. [x] Pipeline de preprocesamiento
3. [x] Entrenamiento y comparación de modelos
4. [x] Evaluación con métricas de clasificación
5. [x] Validación cruzada
6. [x] Registro de experimentos en MLflow
7. [x] Interpretación de resultados con matrices de confusión

### 📊 Conclusiones Clave

**~> Basándote en los resultados obtenidos:**

1. **¿Qué modelo funcionó mejor?**
   - ~> Random Forest funcionó mejor que SGD en todas las métricas

2. **¿Por qué crees que funcionó mejor?**
   - ~> Random Forest captura relaciones no lineales y reduce varianza con múltiples árboles

3. **¿El modelo es suficientemente bueno para uso médico?**
   - ~> Con Recall ~0.94 es prometedor, pero requiere validación clínica antes de uso real

4. **¿Qué métrica es más importante en este caso?**
   - ~> Recall, porque perder un paciente enfermo (falso negativo) es más grave que un falso positivo

5. **¿Qué mejorarías del modelo?**
   - ~> Optimizar hiperparámetros con GridSearchCV, probar más modelos y hacer feature engineering

---

### 💡 Reflexiones Importantes

#### ⚕️ Sobre Medicina e IA

**~> Reflexiones:**

1. **Falsos Negativos vs Falsos Positivos**
   - ~> En medicina, los falsos negativos son más graves porque un paciente enfermo no recibiría tratamiento

2. **Responsabilidad Ética**
   - ~> Un modelo de IA no puede tomar decisiones médicas solo, siempre debe ser una herramienta de apoyo al criterio médico

3. **Interpretabilidad**
   - ~> Es importante que los médicos entiendan cómo decide el modelo para confiar en las predicciones

---

### 🚀 Desafíos Adicionales

1. **Optimización de Hiperparámetros** con `GridSearchCV`.
2. **Prueba Otros Modelos** como Gradient Boosting, SVM o Logistic Regression.
3. **Feature Engineering** para crear o seleccionar características.
4. **Análisis de Errores** para entender pacientes mal clasificados.
5. **Curva ROC** para comparar thresholds y AUC.


In [0]:
# DESAFÍOS OPCIONALES
# Usa este espacio para completar los desafíos del proyecto.

# ========================================
# -> DESAFÍO 1 - Grid Search
# ========================================
from sklearn.model_selection import GridSearchCV

# ~> Definir el grid de hiperparámetros
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'max_features': [2, 5, 'sqrt', 'log2']
}

mlflow.sklearn.autolog()

with mlflow.start_run(run_name="Grid Search - Random Forest") as run:
    # > Ejecutar GridSearchCV
    grid_search = GridSearchCV(
        RandomForestClassifier(random_state=42),
        param_grid,
        cv=5,
        scoring='recall',
        verbose=1,
        n_jobs=-1
    )
    grid_search.fit(x_train_pr, y_train)
    
    # >> Mostrar mejores parámetros y score
    print("=" * 60)
    print("RESULTADOS DEL GRID SEARCH")
    print("=" * 60)
    print(f"Mejores parámetros: {grid_search.best_params_}")
    print(f"Mejor recall (CV): {grid_search.best_score_:.4f}")
    print("=" * 60)
    
    mlflow.log_param("best_n_estimators", grid_search.best_params_['n_estimators'])
    mlflow.log_param("best_max_depth", grid_search.best_params_['max_depth'])
    mlflow.log_param("best_max_features", grid_search.best_params_['max_features'])
    mlflow.log_metric("best_cv_recall", grid_search.best_score_)

mlflow.end_run()

# ========================================
# -> DESAFÍO 2 - Curva ROC
# ========================================
from sklearn.metrics import roc_curve, auc

# ~> Obtener probabilidades de predicción
y_proba = forest_clf.predict_proba(x_test_pr)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc_val = auc(fpr, tpr)

# > Visualizar la curva ROC
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_val:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('Curva ROC - Random Forest (Test Set)', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f">> AUC en test: {roc_auc_val:.4f}")

# ========================================
# -> DESAFÍO 3 - Importancia de características
# ========================================
# ~> Obtener nombres de las características tras el pipeline
num_features = num_attr
cat_features = list(full_pipeline.named_transformers_['cat'].get_feature_names_out(cat_attr))
feature_names = num_features + cat_features

# ~> Calcular importancia de características del Random Forest
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': forest_clf.feature_importances_
}).sort_values('importance', ascending=True)

# > Visualizar las 15 características más importantes
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feature_importance['feature'].tail(15), feature_importance['importance'].tail(15))
ax.set_xlabel('Importancia', fontsize=11)
ax.set_ylabel('Características', fontsize=11)
ax.set_title('Top 15 Características más Importantes - Random Forest', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\n>> Las características más importantes suelen ser:")
print("   ca, oldpeak, thalach y age (mediciones clínicas clave)")

# ========================================
# -> DESAFÍO 4 - Comparar más modelos
# ========================================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

modelos = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf', random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

mlflow.sklearn.autolog()

print("=" * 60)
print("COMPARACIÓN DE MODELOS")
print("=" * 60)

for nombre, modelo in modelos.items():
    with mlflow.start_run(run_name=nombre):
        # > Entrenar y predecir
        modelo.fit(x_train_pr, y_train)
        preds_model = modelo.predict(x_test_pr)
        acc = accuracy_score(y_test, preds_model)
        rec = recall_score(y_test, preds_model)
        f1_model = f1_score(y_test, preds_model)
        
        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_recall", rec)
        mlflow.log_metric("test_f1", f1_model)
        
        print(f"   {nombre:25s} -> Accuracy: {acc:.4f} | Recall: {rec:.4f} | F1: {f1_model:.4f}")
    
    mlflow.end_run()

print("=" * 60)
print("=> El mejor modelo se evalúa priorizando Recall en contexto médico")
